# Feature Engineering and Integration

### 1. Importing libraries 

In [1]:
import pandas as pd
import numpy as np

### 2. Loading Datasets

In [2]:
accounts=pd.read_csv('../data/ravenstack_accounts.csv')
subscriptions=pd.read_csv('../data/ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('../data/ravenstack_feature_usage.csv')
support_tickets = pd.read_csv('../data/ravenstack_support_tickets.csv')
churn_events = pd.read_csv('../data/ravenstack_churn_events.csv') 

### 3. Defining Functions

In [11]:
# 1. Convert columns to datetime
def convert_to_datetime(df, columns):
    for col in columns:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
    return df

In [4]:
# 2. Create date-based features
def create_date_features(df, date_col, prefix):
    df[f'{prefix}_year'] = df[date_col].dt.year
    df[f'{prefix}_month'] = df[date_col].dt.month
    df[f'{prefix}_day'] = df[date_col].dt.day
    df[f'{prefix}_weekday'] = df[date_col].dt.weekday
    return df

In [5]:
# 3. Aggregate numeric features by group
def aggregate_numeric(df, group_col, agg_dict, suffix=""):
    agg_df = df.groupby(group_col).agg(agg_dict)
    # Flatten MultiIndex columns if any
    agg_df.columns = [f"{col}{suffix}" for col in agg_df.columns]
    agg_df = agg_df.reset_index()
    return agg_df

In [6]:
# 4. Merge datasets with check
def merge_df(left, right, on, how='left'):
    merged = left.merge(right, on=on, how=how)
    print(f"Merged {left.shape} + {right.shape} -> {merged.shape}")
    return merged

In [7]:
# 5. Create subscription duration and active status
def create_subscription_features(sub_df, current_date=pd.Timestamp.today()):
    sub_df['is_active'] = sub_df['end_date'].isna().astype(int)
    sub_df['subscription_duration_days'] = (sub_df['end_date'].fillna(current_date) - sub_df['start_date']).dt.days
    sub_df['avg_mrr_per_seat'] = sub_df['mrr_amount'] / sub_df['seats']
    sub_df['revenue_ratio'] = sub_df['arr_amount'] / sub_df['mrr_amount']
    return sub_df

In [8]:
# 6. Fill missing satisfaction scores for support tickets
def fill_missing_satisfaction(df):
    df['satisfaction_score'] = df['satisfaction_score'].fillna(df['satisfaction_score'].mean())
    return df

In [9]:
# 7. Create churn flag for accounts
def create_churn_flag(accounts_df, churn_df):
    accounts_df['has_churned'] = accounts_df['account_id'].isin(churn_df['account_id']).astype(int)
    return accounts_df

### 4. Preprocess Data Columns

In [12]:
accounts = convert_to_datetime(accounts, ['signup_date'])
subscriptions = convert_to_datetime(subscriptions, ['start_date', 'end_date'])
support_tickets = convert_to_datetime(support_tickets, ['submitted_at', 'closed_at'])
feature_usage = convert_to_datetime(feature_usage, ['usage_date'])
churn_events = convert_to_datetime(churn_events, ['churn_date'])

### 5. Feature Engineering

In [13]:
# Accounts
accounts = create_date_features(accounts, 'signup_date', 'signup')

In [14]:
# Subscriptions
subscriptions = create_subscription_features(subscriptions)

In [15]:
# Support Tickets - aggregate per account
support_agg = aggregate_numeric(
    support_tickets, 
    'account_id',
    {
        'resolution_time_hours':'mean',
        'first_response_time_minutes':'mean',
        'satisfaction_score':'mean',
        'ticket_id':'count',
        'escalation_flag':'sum'
    },
    suffix="_ticket"
)
support_agg.rename(columns={'ticket_id_ticket':'total_tickets','escalation_flag_ticket':'num_escalated_tickets'}, inplace=True)

In [16]:
# Feature Usage - aggregate per subscription
feature_agg = aggregate_numeric(
    feature_usage,
    'subscription_id',
    {
        'usage_count':'sum',
        'usage_duration_secs':'sum',
        'error_count':'sum',
        'is_beta_feature':'sum'
    },
    suffix="_feat"
)
feature_agg['beta_feature_usage_ratio'] = feature_agg['is_beta_feature_feat'] / feature_agg['usage_count_feat']


In [17]:
# Churn Events - for accounts
churn_events = churn_events.copy()  # ensure safe operations
accounts = create_churn_flag(accounts, churn_events)

### 6. Merge Datasets

In [18]:
# Merge subscriptions with accounts
data = merge_df(subscriptions, accounts, 'account_id')


Merged (5000, 18) + (500, 15) -> (5000, 32)


In [19]:
# Merge feature usage aggregated
data = merge_df(data, feature_agg, 'subscription_id')


Merged (5000, 32) + (4967, 6) -> (5000, 37)


In [20]:
# Merge support tickets aggregated
data = merge_df(data, support_agg, 'account_id')

Merged (5000, 37) + (492, 6) -> (5000, 42)


In [21]:
# Merge churn events (optional detailed info)
data = merge_df(data, churn_events[['account_id','reason_code','refund_amount_usd','preceding_upgrade_flag','preceding_downgrade_flag','is_reactivation']], 'account_id')

Merged (5000, 42) + (600, 6) -> (7429, 47)


### BACKUP

In [30]:
# Backup
data_backup = data.copy()
print("Backup saved to data_backup (in-memory).")

Backup saved to data_backup (in-memory).


### 7. Duplicate Inspection and Resolution

#### a. Inspection

In [25]:
# A. quick counts
print("Total rows:", data.shape[0])
print("Exact duplicate rows (identical rows):", data.duplicated().sum())

Total rows: 7429
Exact duplicate rows (identical rows): 205


In [26]:
# B. show a few exact duplicate rows (if any)
exact_dups = data[data.duplicated(keep=False)]
print("Sample exact duplicate rows (if any):")
display(exact_dups.head(10))


Sample exact duplicate rows (if any):


,subscription_id,account_id,start_date,end_date,plan_tier_x,seats_x,mrr_amount,arr_amount,is_trial_x,upgrade_flag,...,resolution_time_hours_ticket,first_response_time_minutes_ticket,satisfaction_score_ticket,total_tickets,num_escalated_tickets,reason_code,refund_amount_usd,preceding_upgrade_flag,preceding_downgrade_flag,is_reactivation
59,S-73a4e4,A-993085,2024-09-09,0,Pro,31,1519,18228,False,False,...,46.500000,82.00,3.500000,4.0,0.0,features,0.0,False,False,False
60,S-73a4e4,A-993085,2024-09-09,0,Pro,31,1519,18228,False,False,...,46.500000,82.00,3.500000,4.0,0.0,features,0.0,False,False,False
78,S-b9fc6c,A-2ebb4b,2023-06-16,0,Basic,41,779,9348,False,False,...,43.500000,136.50,3.000000,2.0,0.0,pricing,0.0,False,False,False
79,S-b9fc6c,A-2ebb4b,2023-06-16,0,Basic,41,779,9348,False,False,...,43.500000,136.50,3.000000,2.0,0.0,pricing,0.0,False,False,False
92,S-f5c472,A-faa28c,2024-07-09,0,Basic,48,912,10944,False,False,...,52.250000,115.50,4.666667,4.0,1.0,features,0.0,False,False,False
93,S-f5c472,A-faa28c,2024-07-09,0,Basic,48,912,10944,False,False,...,52.250000,115.50,4.666667,4.0,1.0,features,0.0,False,False,False
162,S-628729,A-157070,2024-10-05,0,Enterprise,23,4577,54924,False,False,...,42.333333,118.00,3.000000,3.0,1.0,features,0.0,False,False,False
163,S-628729,A-157070,2024-10-05,0,Enterprise,23,4577,54924,False,False,...,42.333333,118.00,3.000000,3.0,1.0,features,0.0,False,False,False
166,S-86455a,A-1b9609,2023-12-01,0,Pro,12,588,7056,False,False,...,43.000000,74.75,3.000000,4.0,0.0,unknown,0.0,False,False,False
168,S-86455a,A-1b9609,2023-12-01,0,Pro,12,588,7056,False,False,...,43.000000,74.75,3.000000,4.0,0.0,unknown,0.0,False,False,False


In [27]:
# C. duplicates by subscription_id
dup_by_sub = data.groupby('subscription_id').size().reset_index(name='count').query('count>1').sort_values('count', ascending=False)
print("Subscriptions with >1 rows (top 10):")
display(dup_by_sub.head(10))


Subscriptions with >1 rows (top 10):


,subscription_id,count
4523,S-e7a190,5
1696,S-561850,5
3465,S-b262a1,5
4892,S-fa05d7,5
4217,S-d8c527,5
3850,S-c62bb7,5
4552,S-e9395e,5
3221,S-a5a40b,5
2706,S-8b0498,5
2613,S-85f2ab,5


In [31]:
# D. duplicates by account_id (useful to see multiplicative effect)
dup_by_acc = data.groupby('account_id').size().reset_index(name='count').sort_values('count', ascending=False)
print("Accounts with most rows (top 10):")
display(dup_by_acc.head(10))

Accounts with most rows (top 10):


,account_id,count
30,A-0cc442,52
467,A-ee42db,48
268,A-87aa7b,48
212,A-6da850,48
5,A-029f69,48
204,A-68f37c,48
50,A-180abf,45
220,A-712426,45
126,A-425e76,44
167,A-533452,44


#### b. Resolution

In [33]:
# 1) Drop exact identical duplicates (easy win)
print("\n1) Dropping exact identical duplicates...")
before = data.shape[0]
data = data.drop_duplicates()
after = data.shape[0]
print(f"Dropped {before-after} exact identical rows. New shape: {data.shape}")

# 2) Inspect duplicates by subscription_id and account_id
print("\n2) Inspect duplicates by subscription_id and account_id...")
dup_by_sub = data.groupby('subscription_id').size().reset_index(name='count').query('count>1').sort_values('count', ascending=False)
dup_by_acc = data.groupby('account_id').size().reset_index(name='count').sort_values('count', ascending=False)

print("Subscriptions with >1 rows (top 10):")
display(dup_by_sub.head(10))
print("Accounts with most rows (top 10):")
display(dup_by_acc.head(10))

if dup_by_sub.empty:
    print("\nNo subscription-level duplicates remain. Proceed to final sanity checks.")
else:
    # show sample duplicate subscription for inspection
    sample_sub = dup_by_sub['subscription_id'].iloc[0]
    print(f"\nSample duplicated subscription_id (showing rows): {sample_sub}")
    display(data[data['subscription_id']==sample_sub])
    
    # find churn-related columns if present
    churn_cols = [c for c in ['churn_event_id','churn_date','reason_code','refund_amount_usd',
                              'preceding_upgrade_flag','preceding_downgrade_flag','is_reactivation','feedback_text'] if c in data.columns]
    print("Churn-related columns in main data:", churn_cols)
    
    # If churn columns are present in merged data, aggregate churn_events by account and re-merge
    if churn_cols and 'churn_events' in globals():
        print("\nAggregating churn_events by account_id and re-merging to avoid multiplicative joins...")
        churn_agg = churn_events.groupby('account_id').agg(
            total_churn_events = ('churn_event_id','count'),
            last_churn_date = ('churn_date','max'),
            total_refunds = ('refund_amount_usd','sum'),
            any_reactivation = ('is_reactivation','max'),
            any_preceding_upgrade = ('preceding_upgrade_flag','max'),
            any_preceding_downgrade = ('preceding_downgrade_flag','max'),
            top_reason = ('reason_code', lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
        ).reset_index()

        # drop raw churn cols from main data if present, then merge aggregated
        cols_to_drop = [c for c in churn_cols + ['refund_amount_usd'] if c in data.columns]
        if cols_to_drop:
            print("Dropping churn-related raw cols from main data:", cols_to_drop)
            data = data.drop(columns=cols_to_drop, errors=True)
        data = data.merge(churn_agg, on='account_id', how='left')
        print("Re-merged aggregated churn info. Shape now:", data.shape)
        # recompute dup_by_sub
        dup_by_sub = data.groupby('subscription_id').size().reset_index(name='count').query('count>1').sort_values('count', ascending=False)
        print("Subscriptions with >1 rows after churn aggregation (top 10):")
        display(dup_by_sub.head(10))
    else:
        print("\nNo churn columns found or churn_events not available; skipping churn aggregation.")

    # If duplicates still exist, aggregate by subscription_id with sensible rules
    if not dup_by_sub.empty:
        print("\nAggregating remaining duplicates per subscription_id with sensible rules...")
        # define aggregation plan dynamically (only include columns that exist)
        agg_funcs = {}
        # sums
        for c in ['usage_count_feat','usage_duration_secs_feat','error_count_feat','total_tickets','num_escalated_tickets','total_refunds','mrr_amount','arr_amount']:
            if c in data.columns:
                agg_funcs[c] = 'sum'
        # means
        for c in ['avg_mrr_per_seat','subscription_duration_days','resolution_time_hours_ticket','first_response_time_minutes_ticket','satisfaction_score_ticket','beta_feature_usage_ratio']:
            if c in data.columns:
                agg_funcs[c] = 'mean'
        # categorical -> last
        for c in ['plan_tier_x','plan_tier_y','reason_code','account_name','industry','country','billing_frequency','top_reason']:
            if c in data.columns:
                agg_funcs[c] = 'last'
        # booleans -> max
        for c in ['is_active','has_churned','is_beta_feature_feat','is_trial_x','upgrade_flag','downgrade_flag','churn_flag_x','auto_renew_flag','any_reactivation','any_preceding_upgrade','any_preceding_downgrade']:
            if c in data.columns:
                agg_funcs[c] = 'max'

        print("Aggregation rules will be applied to these columns:")
        display(pd.Series(agg_funcs).head(60))

        grouped = data.groupby('subscription_id').agg(agg_funcs).reset_index()

        keep_cols = ['account_id', 'start_date', 'end_date']   # do NOT include 'subscription_id' here
        available_keep = [c for c in keep_cols if c in data.columns]

        if available_keep:
            firsts = data.groupby('subscription_id')[available_keep].first().reset_index()
        else:
            # fallback: create a simple subscription_id list if none of the keep columns exist
            firsts = data[['subscription_id']].drop_duplicates().reset_index(drop=True)

        data = firsts.merge(grouped, on='subscription_id', how='left')
        print("Aggregation done. New shape:", data.shape)

# Final quick checks after de-duping
print("\n--- Post-dedupe checks ---")
print("Total rows:", data.shape[0])
print("Exact duplicate rows remaining:", data.duplicated().sum())
if 'subscription_id' in data.columns:
    max_per_sub = data.groupby('subscription_id').size().max()
    print("Max rows per subscription_id:", max_per_sub)
print("Done. Inspect the top rows if needed:")
display(data.head())



1) Dropping exact identical duplicates...
Dropped 2224 exact identical rows. New shape: (5000, 49)

2) Inspect duplicates by subscription_id and account_id...
Subscriptions with >1 rows (top 10):


,subscription_id,count


Accounts with most rows (top 10):


,account_id,count
225,A-726cfa,19
418,A-d4ac0e,19
178,A-592832,19
183,A-5a92e7,19
181,A-5a215a,18
277,A-8bde0c,18
259,A-82d8a6,17
217,A-6f8ad2,17
205,A-692f18,17
268,A-87aa7b,16



No subscription-level duplicates remain. Proceed to final sanity checks.

--- Post-dedupe checks ---
Total rows: 5000
Exact duplicate rows remaining: 0
Max rows per subscription_id: 1
Done. Inspect the top rows if needed:


,subscription_id,account_id,start_date,end_date,plan_tier_x,seats_x,mrr_amount,arr_amount,is_trial_x,upgrade_flag,...,satisfaction_score_ticket,total_tickets,num_escalated_tickets,total_churn_events,last_churn_date,total_refunds,any_reactivation,any_preceding_upgrade,any_preceding_downgrade,top_reason
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12 00:00:00,Enterprise,14,2786,33432,False,False,...,3.500000,4.0,0.0,NaN,NaT,NaN,NaN,NaN,NaN,NaN
1,S-0f6f44,A-9b9fe9,2024-06-11,0,Pro,17,833,9996,False,False,...,4.000000,4.0,0.0,2.0,2024-09-04,27.92,False,False,False,features
3,S-51c0d1,A-659280,2024-11-25,0,Enterprise,62,0,0,True,True,...,4.333333,4.0,0.0,2.0,2024-11-22,27.75,False,False,False,features
5,S-f81687,A-e7a1e2,2024-11-23,2024-12-13 00:00:00,Enterprise,5,995,11940,False,False,...,4.000000,1.0,0.0,3.0,2024-12-11,6.20,True,True,False,budget
8,S-cff5a2,A-ba6516,2024-01-10,0,Enterprise,27,5373,64476,False,False,...,4.000000,6.0,0.0,1.0,2024-03-17,0.00,False,False,False,budget


In [35]:
# Quick checks: dtypes, missing summary, sample counts
print("Shape:", data.shape)
print("\nDtypes:\n", data.dtypes)
print("\nMissing values (top 20):")
print(data.isnull().sum().sort_values(ascending=False).head(20))

# Check for non-datetime values masquerading as dates (e.g., '0' or strings)
for date_col in ['start_date','end_date','signup_date','last_churn_date']:
    if date_col in data.columns:
        non_dt = data[~pd.to_datetime(data[date_col], errors='coerce').notna()][date_col]
        print(f"\n{date_col} - count non-parsable values:", non_dt.shape[0])
        if non_dt.shape[0] > 0:
            display(non_dt.head(5))

# Confirm one row per subscription_id
if 'subscription_id' in data.columns:
    max_per_sub = data.groupby('subscription_id').size().max()
    print("\nMax rows per subscription_id:", max_per_sub)

Shape: (5000, 49)

Dtypes:
 subscription_id                               object
account_id                                    object
start_date                            datetime64[ns]
end_date                                      object
plan_tier_x                                   object
seats_x                                        int64
mrr_amount                                     int64
arr_amount                                     int64
is_trial_x                                      bool
upgrade_flag                                    bool
downgrade_flag                                  bool
churn_flag_x                                    bool
billing_frequency                             object
auto_renew_flag                                 bool
is_active                                      int64
subscription_duration_days                     int64
avg_mrr_per_seat                             float64
revenue_ratio                                float64
account_name      

0    NaT
16   NaT
18   NaT
38   NaT
44   NaT
Name: last_churn_date, dtype: datetime64[ns]


Max rows per subscription_id: 1


#### c. Fix churn related columns

In [40]:
from IPython.display import display

# 1) Numeric churn aggregates -> fill NaN with 0
num_fill_zero = ['total_churn_events','total_refunds']
for c in num_fill_zero:
    if c in data.columns:
        data[c] = pd.to_numeric(data[c], errors='coerce').fillna(0)

# 2) Boolean flags from churn aggregation -> fill NaN with False and coerce to bool
bool_cols = ['any_reactivation','any_preceding_upgrade','any_preceding_downgrade','has_churned']
for c in bool_cols:
    if c in data.columns:
        # map common truthy string/int forms to booleans, then fillna(False) and cast
        data[c] = data[c].map({True:True, False:False, 'True':True, 'False':False, 'true':True, 'false':False, 1:True, 0:False}).fillna(False).astype(bool)

# 3) top_reason: fill missing with a sensible label
if 'top_reason' in data.columns:
    data['top_reason'] = data['top_reason'].fillna('no_churn').astype(str)

# 4) last_churn_date: keep NaT for non-churned (no change), but ensure dtype is datetime
if 'last_churn_date' in data.columns:
    data['last_churn_date'] = pd.to_datetime(data['last_churn_date'], errors='coerce', dayfirst=True)

# 5) Recompute any dependent fields (optional)
if 'total_churn_events' in data.columns:
    # ensure integer counts
    data['total_churn_events'] = data['total_churn_events'].astype(int)

# 6) Final quick checks after fixes
print("Dtypes (sample churn-related):")
display(data[[c for c in ['total_churn_events','total_refunds','any_reactivation',
                         'any_preceding_upgrade','any_preceding_downgrade','top_reason',
                         'last_churn_date'] if c in data.columns]].dtypes)

print("\nMissing values (top 20):")
display(data.isnull().sum().sort_values(ascending=False).head(20))

print("\nValue counts for boolean churn flags (example):")
for c in ['any_reactivation','any_preceding_upgrade','any_preceding_downgrade','has_churned']:
    if c in data.columns:
        print(f"\n{c} value counts:")
        display(data[c].value_counts(dropna=False))


Dtypes (sample churn-related):


total_churn_events                  int64
total_refunds                     float64
any_reactivation                     bool
any_preceding_upgrade                bool
any_preceding_downgrade              bool
top_reason                         object
last_churn_date            datetime64[ns]
dtype: object


Missing values (top 20):


last_churn_date               1472
account_id                       0
start_date                       0
end_date                         0
plan_tier_x                      0
seats_x                          0
mrr_amount                       0
arr_amount                       0
is_trial_x                       0
upgrade_flag                     0
downgrade_flag                   0
churn_flag_x                     0
billing_frequency                0
auto_renew_flag                  0
is_active                        0
subscription_duration_days       0
subscription_id                  0
avg_mrr_per_seat                 0
revenue_ratio                    0
industry                         0
dtype: int64


Value counts for boolean churn flags (example):

any_reactivation value counts:


...

any_preceding_upgrade
False    3907
True     1093
Name: count, dtype: int64


any_preceding_downgrade value counts:


any_preceding_downgrade
False    4535
True      465
Name: count, dtype: int64


has_churned value counts:


has_churned
True     3528
False    1472
Name: count, dtype: int64

#### d. Quick Sanity Checks

In [41]:
# 1) Inspect a few subscriptions that had churn (has_churned True) and some that didn't
print("Sample churned subscriptions (head):")
display(data[data['has_churned']==True].head(5)[['subscription_id','account_id','total_churn_events','last_churn_date','total_refunds','top_reason']])

print("\nSample non-churned subscriptions (head):")
display(data[data['has_churned']==False].head(5)[['subscription_id','account_id','total_churn_events','last_churn_date','total_refunds','top_reason']])

# 2) Check refunds distribution (sanity)
if 'total_refunds' in data.columns:
    print("\nRefunds stats:")
    display(data['total_refunds'].describe())

# 3) Verify a previously problematic subscription id (replace with one you noted earlier)
sample_ids = ['S-c62bb7','S-e7a190','S-561850']  # example; change to ones you saw earlier
for s in sample_ids:
    if s in data['subscription_id'].values:
        print(f"\nDetails for {s}:")
        display(data[data['subscription_id']==s].T)
    else:
        print(f"\n{s} not present in final data.")

data = data.rename(columns={'plan_tier_x':'plan_tier_subscription','plan_tier_y':'plan_tier_account',
                            'seats_x':'seats_subscription','seats_y':'seats_account'})



Sample churned subscriptions (head):


,subscription_id,account_id,total_churn_events,last_churn_date,total_refunds,top_reason
1,S-0f6f44,A-9b9fe9,2,2024-09-04,27.92,features
3,S-51c0d1,A-659280,2,2024-11-22,27.75,features
5,S-f81687,A-e7a1e2,3,2024-12-11,6.20,budget
8,S-cff5a2,A-ba6516,1,2024-03-17,0.00,budget
9,S-4b9b13,A-fa2041,1,2024-07-13,0.00,competitor



Sample non-churned subscriptions (head):


,subscription_id,account_id,total_churn_events,last_churn_date,total_refunds,top_reason
0,S-8cec59,A-3c1a3f,0,NaT,0.0,no_churn
16,S-79a9e1,A-3a5ad8,0,NaT,0.0,no_churn
18,S-7acd8b,A-7920cc,0,NaT,0.0,no_churn
38,S-c07ede,A-cea1f0,0,NaT,0.0,no_churn
44,S-c75159,A-bcf87c,0,NaT,0.0,no_churn



Refunds stats:


...

,5982
subscription_id,S-c62bb7
account_id,A-0a62f5
start_date,2024-09-16 00:00:00
end_date,0
plan_tier_x,Enterprise
seats_x,16
mrr_amount,3184
arr_amount,38208
is_trial_x,False
upgrade_flag,False



Details for S-e7a190:


,5223
subscription_id,S-e7a190
account_id,A-0a62f5
start_date,2024-12-06 00:00:00
end_date,0
plan_tier_x,Basic
seats_x,22
mrr_amount,418
arr_amount,5016
is_trial_x,False
upgrade_flag,False



Details for S-561850:


,1111
subscription_id,S-561850
account_id,A-0a62f5
start_date,2024-10-23 00:00:00
end_date,0
plan_tier_x,Pro
seats_x,16
mrr_amount,784
arr_amount,9408
is_trial_x,False
upgrade_flag,False


### 8. Final Integrated Dataset

In [44]:
# Save final dataset
out_path = "../data/final_integrated_dataset.csv"
import os
os.makedirs(os.path.dirname(out_path), exist_ok=True)
data.to_csv(out_path, index=False)
print("Saved final dataset to:", out_path)

# Print completion message
print("Phase 3: Feature Engineering & Integration Completed! ")

Saved final dataset to: ../data/final_integrated_dataset.csv
Phase 3: Feature Engineering & Integration Completed! 
